In [59]:
##### Final cleaning of RF training data before runs
# convert to log, remove outliers, add country codes 

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [60]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent.parent 

# data 
capital_abs = pd.read_csv(f'{cd}/Data/Clean/Training_data/capital_absolute.csv')
labor_abs = pd.read_csv(f'{cd}/Data/Clean/Training_data/labor_absolute.csv')

capital_rtv = pd.read_csv(f'{cd}/Data/Clean/Training_data/capital_relative.csv')
labor_rtv = pd.read_csv(f'{cd}/Data/Clean/Training_data/labor_relative.csv')

# save paths
save_path_capital_abs = f'{cd}/Data/Clean/Training_data/capital_absolute_final.csv'
save_path_labor_abs = f'{cd}/Data/Clean/Training_data/labor_absolute_final.csv'

save_path_capital_rtv = f'{cd}/Data/Clean/Training_data/capital_relative_final.csv'
save_path_labor_rtv = f'{cd}/Data/Clean/Training_data/labor_relative_final.csv'

In [61]:
##### Variables 

### Drop aggregate variables

capital_abs = capital_abs.drop(columns = ['GEO_ID_NAME', 'ag_capital_stock_USD_nominal', 'total_production_USD', 'total_production_t'])
labor_abs = labor_abs.drop(columns = ['GEO_ID_NAME', 'ag_jobs', 'total_production_USD', 'total_production_t'])

In [62]:
##### Drop regions with missing data 

c_len_pre = len(capital_abs)
capital_abs = capital_abs.dropna()
c_len_pos = len(capital_abs)
c_dropped = c_len_pre - c_len_pos

print(f'Capital absolute: {c_dropped} out of {c_len_pre} dropped')

c_len_pre = len(capital_rtv)
capital_rtv = capital_rtv.dropna()
c_len_pos = len(capital_rtv)
c_dropped = c_len_pre - c_len_pos

print(f'Capital relative: {c_dropped} out of {c_len_pre} dropped')

l_len_pre = len(labor_abs)
labor_abs = labor_abs.dropna()
l_len_pos = len(labor_abs)
l_dropped = l_len_pre - l_len_pos

print(f'Labor absolute: {l_dropped} out of {l_len_pre} dropped')

l_len_pre = len(labor_rtv)
labor_rtv = labor_rtv.dropna()
l_len_pos = len(labor_rtv)
l_dropped = l_len_pre - l_len_pos

print(f'Labor relative: {l_dropped} out of {l_len_pre} dropped')

Capital absolute: 371 out of 10482 dropped
Capital relative: 371 out of 10482 dropped
Labor absolute: 469 out of 11500 dropped
Labor relative: 465 out of 11500 dropped


In [63]:
##### Add country ID column

capital_abs['country_ID'] = capital_abs['PROJ_ID'].str[:3]
labor_abs['country_ID'] = labor_abs['PROJ_ID'].str[:3]

capital_rtv['country_ID'] = capital_rtv['PROJ_ID'].str[:3]
labor_rtv['country_ID'] = labor_rtv['PROJ_ID'].str[:3]

In [64]:
##### Convert intensities to log for absolute model 

#### rescale variables before transformation

## capital intensities: 
# USD per USD -> USD per million USD
capital_abs['capital_intensity_USD_per_million_USD'] = capital_abs['capital_intensity_USD_per_USD'] * 1e6
capital_abs['country_capital_intensity_USD_per_million_USD'] = capital_abs['country_capital_intensity_USD_per_USD'] * 1e6
labor_abs['country_capital_intensity_USD_per_million_USD'] = labor_abs['country_capital_intensity_USD_per_USD'] * 1e6
# USD per tonne -> USD per million tonne
capital_abs['capital_intensity_USD_per_million_tonne'] = capital_abs['capital_intensity_USD_per_tonne'] * 1e6
capital_abs['country_capital_intensity_USD_per_million_tonne'] = capital_abs['country_capital_intensity_USD_per_tonne'] * 1e6
labor_abs['country_capital_intensity_USD_per_million_tonne'] = labor_abs['country_capital_intensity_USD_per_tonne'] * 1e6

## labor intensities: 
# jobs per million USD -> stays the same
# jobs per tonne -> jobs per million tonnes
labor_abs['labor_intensity_jobs_per_million_tonne'] = labor_abs['labor_intensity_jobs_per_tonne'] * 1e6
labor_abs['country_labor_intensity_jobs_per_million_tonne'] = labor_abs['country_labor_intensity_jobs_per_tonne'] * 1e6
capital_abs['country_labor_intensity_jobs_per_million_tonne'] = capital_abs['country_labor_intensity_jobs_per_tonne'] * 1e6

## production intensities:
# USD production per HA -> USD production per million HA
capital_abs['USD_production_per_million_HA'] = capital_abs['USD_production_per_HA'] * 1e6
labor_abs['USD_production_per_million_HA'] = labor_abs['USD_production_per_HA'] * 1e6
# tonnes production per HA -> tonnes production per million HA
capital_abs['tonnes_production_per_million_HA'] = capital_abs['tonnes_production_per_HA'] * 1e6
labor_abs['tonnes_production_per_million_HA'] = labor_abs['tonnes_production_per_HA'] * 1e6

# transform
def log_transform(df, cols):
    for col in cols:
        df[f"log_{col}"] = np.log1p(df[col])
    df.drop(columns=cols, inplace=True)
    return df

capital_cols = [
    "capital_intensity_USD_per_million_USD",
    "capital_intensity_USD_per_million_tonne",
    "tonnes_production_per_million_HA",
    "USD_production_per_million_HA",
    "country_capital_intensity_USD_per_million_USD",
    "country_labor_intensity_jobs_per_million_USD",
    "country_capital_intensity_USD_per_million_tonne",
    "country_labor_intensity_jobs_per_million_tonne"
]

labor_cols = [
    "labor_intensity_jobs_per_million_USD",
    "labor_intensity_jobs_per_million_tonne",
    "tonnes_production_per_million_HA",
    "USD_production_per_million_HA",
    "country_labor_intensity_jobs_per_million_USD",
    "country_capital_intensity_USD_per_million_USD",
    "country_capital_intensity_USD_per_million_tonne",
    "country_labor_intensity_jobs_per_million_tonne"
]

capital_abs = log_transform(capital_abs, capital_cols)
labor_abs = log_transform(labor_abs, labor_cols)

#### Drop old columns
cols_to_drop = [
    'capital_intensity_USD_per_USD',
    'country_capital_intensity_USD_per_USD',
    'capital_intensity_USD_per_tonne',
    'country_capital_intensity_USD_per_tonne',
    'labor_intensity_jobs_per_tonne',
    'country_labor_intensity_jobs_per_tonne',
    'USD_production_per_HA',
    'tonnes_production_per_HA'
]

# Drop columns
for df in [capital_abs, labor_abs]:
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

In [65]:
##### Drop countries 

# capital
# countries where a proxy for capital was used 'BRA', 'EGY', 'GHA', 'IND', 'TUR', 'ZAF', 'TZA', 'ARG'
# 'ZAF', 'IND: dropped because proxy for capital was used and produced some wierd values
# 'BGR', 'SWE', 'HUN', 'FIN', 'POL': dropped because negative R2 across both test and train, USD and tonne

cap_countries = ['ZAF', 'IND', 'BGR', 'SWE', 'HUN', 'FIN', 'POL'] 
capital_abs = capital_abs[~capital_abs['country_ID'].isin(cap_countries)]
capital_rtv = capital_rtv[~capital_rtv['country_ID'].isin(cap_countries)]

# labor 
# 'BEL', 'HUN', 'ROU', 'FIN': dropped because negative R2 across both test and train, USD and tonne

lab_countries = ['BEL', 'HUN', 'ROU', 'FIN'] 
labor_abs = labor_abs[~labor_abs['country_ID'].isin(lab_countries)]
labor_rtv = labor_rtv[~labor_rtv['country_ID'].isin(lab_countries)]


In [66]:
#####  Drop extreme values (top/bottom 1%)

def drop_extremes (df, var_1, var_2):
    
    lower_cap_1 = df[var_1].quantile(0.01)
    upper_cap_1 = df[var_1].quantile(0.99)

    lower_cap_2 = df[var_2].quantile(0.01)
    upper_cap_2 = df[var_2].quantile(0.99)

    df = df[
        (df[var_1] >= lower_cap_1) &
        (df[var_1] <= upper_cap_1)
    ]

    df = df[
        (df[var_2] >= lower_cap_2) &
        (df[var_2] <= upper_cap_2)
    ]

    return df

capital_abs = drop_extremes(capital_abs, "log_capital_intensity_USD_per_million_USD", "log_capital_intensity_USD_per_million_tonne")
labor_abs = drop_extremes(labor_abs, "log_labor_intensity_jobs_per_million_USD", "log_labor_intensity_jobs_per_million_tonne")

capital_rtv = drop_extremes(capital_rtv, "rtv_log_capital_intensity_USD_per_million_USD", "rtv_log_capital_intensity_USD_per_million_tonne")
labor_rtv = drop_extremes(labor_rtv, "rtv_log_labor_intensity_jobs_per_million_tonne", "rtv_log_labor_intensity_jobs_per_million_USD")

In [67]:
##### Save 
capital_abs.to_csv(save_path_capital_abs, index=False)
labor_abs.to_csv(save_path_labor_abs, index=False)

capital_rtv.to_csv(save_path_capital_rtv, index=False)
labor_rtv.to_csv(save_path_labor_rtv, index=False)